# Lab 06: Why Shadows Look Wrong in Space

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 1, *The Reality Generator*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-06.ipynb)

**What you will do:** judge whether shaded circles look like bumps or craters at three rotations, compare your judgements with the book's predicted pattern, and test whether a real depth-estimation model's 3D reading flips the same way yours does.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 06 you looked at a pattern of shaded circles (or lunar craters, a textured ceiling, or a
coin) and judged which ones looked like bumps and which looked like craters, then rotated the
image (or object) 180 degrees and watched the judgement flip, and rotated it only 90 degrees and
watched the effect weaken.

## Record your results

Test the crater/bump image (or the coin variation) at three orientations: 0 degrees (as
found), 90 degrees, and 180 degrees. For each, note whether you see bumps, craters, or
something ambiguous/flat.

In [ ]:
# example data: replace with your own judgements
shading_log = pd.DataFrame({
    "rotation_deg": [0, 90, 180],
    "judgement": ["bumps", "flat/ambiguous", "craters"],   # example data
    "confidence_1to5": [5, 2, 5],                            # example data
})
display(shading_log)

## Compare

Lab 06 describes a specific pattern: at 0 degrees the shapes should read as bumps; rotating 180
degrees should flip bumps to craters; rotating only 90 degrees should weaken or flatten the
effect, "the circles look flatter, more ambiguous", because the shading gradient no longer
points up or down.

In [ ]:
expected = {0: "bumps", 90: "ambiguous", 180: "craters"}
for _, row in shading_log.iterrows():
    deg = row["rotation_deg"]
    low_confidence_ok = (deg == 90 and row["confidence_1to5"] <= 3)
    matches = (expected[deg] in row["judgement"].lower()) or low_confidence_ok
    print(f"{deg} degrees: you saw '{row['judgement']}' "
          f"({'matches' if matches else 'differs from'} the book's expected pattern)")

## The AI side

The book notes that modern monocular depth-estimation networks such as MiDaS and Depth Anything
"implicitly learn the light-from-above prior from the statistics of their training data" -- and
that their depth estimates "can be inverted" on unusually lit images, just as your own
perception is. Below, a small open Depth Anything model estimates depth for the same
shaded-circle pattern at 0 degrees and rotated 180 degrees.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
from PIL import Image, ImageDraw, ImageFilter
from transformers import pipeline

def shaded_circles(rotation_deg):
    img = Image.new("L", (300, 150), 160)
    draw = ImageDraw.Draw(img)
    for cx in (75, 150, 225):
        for r, shade in [(30, 230), (20, 200), (10, 160)]:
            draw.ellipse([cx - r, 75 - r, cx + r, 75 + r], fill=shade)
    img = img.filter(ImageFilter.GaussianBlur(3)).convert("RGB")
    return img.rotate(rotation_deg, fillcolor=(160, 160, 160))

depth_estimator = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf")

fig, axes = plt.subplots(2, 2, figsize=(7, 6))
for row, rotation in enumerate([0, 180]):
    img = shaded_circles(rotation)
    depth = np.array(depth_estimator(img)["depth"])
    axes[row, 0].imshow(img); axes[row, 0].set_title(f"Input, {rotation} degrees", fontsize=9)
    axes[row, 1].imshow(depth, cmap="inferno"); axes[row, 1].set_title("Depth Anything's estimate", fontsize=9)
    for ax in axes[row]:
        ax.axis("off")
    centre_depth = depth[65:85, 65:85].mean()
    surround_depth = depth[10:30, 10:30].mean()
    print(f"{rotation} degrees: centre depth value {centre_depth:.1f} vs surround {surround_depth:.1f} "
          f"({'centre higher' if centre_depth > surround_depth else 'centre lower'})")
plt.tight_layout(); plt.show()

Depth Anything's output is a relative depth value, not an absolute "nearer/farther" label, so
what matters is whether the relationship between the centre and the surround flips between the
two rotations. If it does, the network is doing exactly what your visual system does: reading
3D shape off the shading gradient with no separate "common sense" check on which way is really
up. The disanalogy is in what the network lacks, not what it shares: it has no light-from-above
rule hard-wired in the way your brain does. Its behaviour comes entirely from the statistics of
the photographs it was trained on, so a different training set could in principle give it the
opposite bias -- whereas your own prior stays stubborn even once you are told the truth.

## Pool the class data

Pool the class's bump/crater judgements at each rotation: each person adds an anonymous ID and
their judgement at 0 and 180 degrees, downloads the results as CSV and loads it here.

In [ ]:
import io
csv_text = """id,judgement_0deg,judgement_180deg
P01,bumps,craters
P02,bumps,craters
P03,bumps,bumps
P04,bumps,craters
P05,craters,bumps
"""  # example data: replace with your class CSV, e.g. pd.read_csv("shading_class.csv")
df = pd.read_csv(io.StringIO(csv_text))
prop_bumps_0 = (df.judgement_0deg == "bumps").mean()
prop_bumps_180 = (df.judgement_180deg == "bumps").mean()

def bootstrap_prop(series, value):
    boot = [(series.sample(len(series), replace=True) == value).mean() for _ in range(5000)]
    return np.percentile(boot, [2.5, 97.5])

ci0 = bootstrap_prop(df.judgement_0deg, "bumps")
ci180 = bootstrap_prop(df.judgement_180deg, "bumps")
print(f"'Bumps' at 0 degrees: {prop_bumps_0:.0%} (95% CI {ci0[0]:.0%}-{ci0[1]:.0%}, n = {len(df)})")
print(f"'Bumps' at 180 degrees: {prop_bumps_180:.0%} (95% CI {ci180[0]:.0%}-{ci180[1]:.0%}, n = {len(df)})")

pd.DataFrame({"0 degrees": [prop_bumps_0], "180 degrees": [prop_bumps_180]}).T.plot.bar(
    legend=False, color=["steelblue", "firebrick"], figsize=(5, 3))
plt.ylabel("proportion reporting 'bumps'"); plt.ylim(0, 1); plt.tight_layout(); plt.show()

## Questions to think about

1. Your judgement at 90 degrees was probably less confident than at 0 or 180. Using the light-from-above prior explanation, why would a shading gradient that runs left-right (rather than up-down) leave the interpretation more ambiguous rather than simply "wrong"?
2. In the Depth Anything demo, did the centre-versus-surround relationship flip between 0 and 180 degrees? If the network's behaviour tracks the shading direction just as reliably as yours does, what does that tell you about whether "light comes from above" needs to be an innate rule, versus something that could be learned purely from the statistics of natural images?
3. The book mentions research suggesting the true prior is "light from above and slightly to the left". Using the coin variation (or the shaded-circle image), test rotating by small amounts, say 10-20 degrees either way from vertical, rather than a clean 90 or 180. Does the bump/crater judgement flip at a different angle when you rotate clockwise versus anticlockwise?
4. Horror films light faces from below specifically because it violates this prior. Using what you now know about the light-from-above assumption, explain why underlighting a face reads as "menacing" rather than simply "unusual".

## Challenge

Photograph a real bumpy or dimpled surface (an orange peel, a golf ball, a stuccoed wall) once as
taken, and again after rotating your phone (or the object) 180 degrees. Judge bump-versus-crater
for both, then flip the photos themselves upside down on a screen and judge again.

In [ ]:
# Example data: replace with your own photographs and judgements.
my_surface_test = pd.DataFrame({
    "condition": ["photo as taken", "phone rotated 180 before shooting", "photo flipped on screen"],
    "judgement": ["bumps", "craters", "craters"],   # example data
})
display(my_surface_test)
print("If rotating the camera before shooting and flipping the finished photo on a screen give")
print("the same judgement, that confirms it is the shading gradient in the final 2D image -- not")
print("anything about the real, physical surface -- that your visual system is reading.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-06.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")